In [ ]:
from langgraph.graph import StateGraph, END, START
from langchain_groq import ChatGroq
from dotenv import load_dotenv
from typing import TypedDict
from langgraph.checkpoint.memory import InMemorySaver


In [ ]:
load_dotenv()

In [ ]:
llm = ChatGroq(model="openai/gpt-oss-20b")

In [ ]:
class JokeState(TypedDict):
    topic: str
    joke: str
    explanation: str

In [ ]:
graph = StateGraph(JokeState)


In [ ]:
def generate_joke(state: JokeState):
    
    prompt = f"Generate a joke on the given topic -> {state['topic']}"
    
    joke = llm.invoke(prompt).content
    
    return {"joke": joke}

In [ ]:
def explain_joke(state: JokeState):

    prompt = f"Explain the given joke : \n {state['joke']}"
    
    explanation = llm.invoke(prompt).content
    
    return {"explanation" : explanation}


In [ ]:
checkpointer = InMemorySaver()
graph.add_node("generate_joke", generate_joke)
graph.add_node("explain_joke", explain_joke)

graph.add_edge(START, "generate_joke")
graph.add_edge("generate_joke", "explain_joke")
graph.add_edge("explain_joke", END)

workflow = graph.compile(checkpointer=checkpointer)
workflow

In [ ]:
thread_id = 1

config = {
    "configurable": {
        "thread_id": thread_id
    }
}
initial_state = {
    "topic": "Cricket"
}

result1 = workflow.invoke(initial_state, config=config)


In [ ]:
result1

In [ ]:
thread_id = 2

config2 = {
    "configurable": {
        "thread_id": thread_id
    }
}
initial_state = {
    "topic": "Science"
}

result2 = workflow.invoke(initial_state, config=config2)

In [ ]:
result2

In [ ]:
workflow.get_state(config={
    "configurable": {
        "thread_id": 2
    }
})

In [ ]:
list(workflow.get_state_history(config=config))

### Time Travel

In [ ]:
workflow.get_state({
    "configurable": {
        "thread_id": 1,
        "checkpoint_id": "1f1d1183-f365-6029-8000-d208569c1e0f"
    }
})

In [ ]:
workflow.invoke(None, {
    "configurable": {
        "thread_id": 1,
        "checkpoint_id": "1f1d1183-f365-6029-8000-d208569c1e0f"
    }
})

In [27]:
## Updating the State at particular checkpoint

workflow.update_state({
    "configurable": {
        "thread_id": 1,
        "checkpoint_id": "1f1d118d-0e5f-6d1e-8001-823a38f26720",
        'checkpoint_ns' : ''
    }
}, {'topic': 'Tennis'})

{'configurable': {'thread_id': '1',
  'checkpoint_ns': '',
  'checkpoint_id': '1f1d1198-cf7e-685d-8002-30c192bb952a'}}

In [28]:
list(workflow.get_state_history(config=config))

[StateSnapshot(values={'topic': 'Tennis'}, next=('generate_joke',), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1d1198-cf7e-685d-8002-30c192bb952a'}}, metadata={'source': 'update', 'step': 2, 'parents': {}}, created_at='2026-10-26T08:45:09.464031+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1d118d-0e5f-6d1e-8001-823a38f26720'}}, tasks=(PregelTask(id='4bbe0601-0d71-b078-20ad-f06d553ed1ab', name='generate_joke', path=('__pregel_pull', 'generate_joke'), error=None, interrupts=(), state=None, result=None),), interrupts=()),
 StateSnapshot(values={'topic': 'Cricket', 'joke': 'Why did the cricket team bring string to the match?\n\nBecause they wanted to *tie* up the score! 🏏😄', 'explanation': '### Why the joke works\n\n| **Part of the joke** | **What it means literally** | **What it means in cricket** | **How the pun is created** |\n|-----------------------|-----------------------------|------------

In [29]:
workflow.invoke(None, {
    "configurable": {
        "thread_id": 1,
        "checkpoint_id": "1f1d1198-cf7e-685d-8002-30c192bb952a"
    }
})

{'topic': 'Tennis',
 'joke': 'Why did the tennis player bring a ladder to the match?  \n\nBecause he heard the game was a real “high‑stakes” sport! 🏆🎾',
 'explanation': '**The joke hinges on a double‑meaning wordplay:**\n\n1. **“High‑stakes”**  \n   - In everyday English, *stakes* refers to the amount of money, prize, or risk involved in a contest.  \n   - When something is described as **“high‑stakes”**, it means the stakes are high—there’s a lot at risk, a big prize, or a big consequence.  \n   - It’s also a common idiom that can be applied to any game or competition, even one that isn’t literally gambling.\n\n2. **A tennis player bringing a ladder**  \n   - A ladder is a literal tool used to reach high places.  \n   - The joke imagines the player interpreting *high‑stakes* literally: the stakes are *high*, so they need a ladder to get *up* to them.\n\n3. **The pun**  \n   - The punchline, “Because he heard the game was a real **‘high‑stakes’** sport!” blends the idiom with the liter